# Spotify dataset — reproducible validation and preprocessing

This notebook runs against the repository's included cleaned dataset by default. It can also preprocess the **original raw dataset** if you supply that file separately.

**Important:** When the cleaned dataset is selected, the notebook **does not reapply outlier capping or other transformations**, because doing so could change the published analysis. The original coursework notebook is preserved as [`original_etl_coursework.ipynb`](original_etl_coursework.ipynb).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Works when launched from the repository root or notebooks/ directory.
root = Path.cwd()
if not (root / 'data' / 'spotify_cleaned.csv').exists():
    root = root.parent
cleaned_path = root / 'data' / 'spotify_cleaned.csv'
assert cleaned_path.is_file(), f'Cannot find {cleaned_path}. Run this notebook from the repository or notebooks folder.'

# Optional: set this to a local copy of the ORIGINAL raw Spotify dataset.
# Example: raw_path = root / 'data' / 'spotify_raw.csv'
raw_path = None
source_path = Path(raw_path) if raw_path is not None else cleaned_path
print('Input:', source_path)
df = pd.read_csv(source_path, encoding='utf-8-sig')
print('Rows:', len(df), '| Columns:', len(df.columns))
df.head()

## 1. Data quality overview

Inspect missing values, duplicates, column types, and the expected columns before making any changes.

In [ ]:
print('Data types:')
print(df.dtypes.to_string())
print('\nMissing values:')
print(df.isna().sum().to_string())
print('\nDuplicate rows:', int(df.duplicated().sum()))
required = {'id', 'name', 'artists', 'popularity', 'release_date'}
missing_required = required - set(df.columns)
assert not missing_required, f'Missing required fields: {sorted(missing_required)}'

## 2. Prepare raw data (only when an original raw file is provided)

The original coursework workflow converted dates and duration, removed the original duration column, and applied IQR-based capping to numeric columns. Those steps are intentionally **not rerun** on the already-cleaned portfolio CSV.

In [ ]:
is_raw = raw_path is not None
if is_raw:
    if 'duration_ms' not in df.columns:
        raise ValueError('Raw preprocessing requires a duration_ms column. Do not use the cleaned CSV as raw input.')
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce', dayfirst=True)
    df['duration_min'] = pd.to_numeric(df['duration_ms'], errors='coerce') / 60000
    df = df.drop(columns=['duration_ms'])

    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if pd.notna(iqr) and iqr > 0:
            df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)
    print('Raw preprocessing completed; review changes before using in Power BI.')
else:
    print('Using published cleaned dataset: no transformations applied.')

## 3. Validation and summary

The summary is computed on the selected dataset. The published cleaned CSV remains unchanged.

In [ ]:
assert 'duration_min' in df.columns, 'Expected duration_min column.'
assert 'Genre' in df.columns, 'Expected Genre column.'
print('Shape:', df.shape)
print('Unique track IDs:', df['id'].nunique(dropna=True))
print('Popularity summary:')
print(pd.to_numeric(df['popularity'], errors='coerce').describe())
print('Duration (minutes) summary:')
print(pd.to_numeric(df['duration_min'], errors='coerce').describe())
print('Missing values (total):', int(df.isna().sum().sum()))

## 4. Optional export

Export only when processing an original raw dataset. This avoids overwriting the portfolio's published CSV by accident.

In [ ]:
if is_raw:
    output_path = root / 'data' / 'spotify_preprocessed_from_raw.csv'
    df.to_csv(output_path, index=False)
    print('Wrote:', output_path)
else:
    print('No export: the published cleaned dataset was inspected without modification.')

## Notes and limitations

- The repository does **not** contain the original raw dataset required to regenerate the exact published CSV from scratch.
- The coursework notebook used a machine-specific Windows input path and contained a final `Genr` assignment sourced from `tempo`; this version avoids that assignment.
- IQR capping can alter values, so the raw-input workflow should be checked against the original coursework methodology before comparing results with the existing dashboard.
- This notebook documents and validates data preparation; it does not rebuild the SQL star schema or Power BI report.